In [1]:
from core import *
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
df = load_data()

In [3]:
#Split data into testing and training
use_reduced_list = True
normalize = True

y = df['trip_duration_seconds']
if not use_reduced_list:
        X = df[['VendorID', 'passenger_count', 'trip_distance', 'payment_type', 
                'flag_no_meter', 'flag_overcharge', 'pickup_hour', 'pickup_dayofweek', 
                'pickup_month', 'is_weekend', 'is_rush_hour', 'store_fwd_flag_int', 
                'ratecode_clean']]
if use_reduced_list:
        X = df[['passenger_count', 'trip_distance', 'payment_type', 
                'pickup_hour', 'pickup_dayofweek', 
                'pickup_month', 'is_weekend', 
        ]]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

if normalize:
        #Normalize features
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train) #Scales the data based on mean/std of training data
        X_test_scaled = scaler.transform(X_test) #Scales the training data in the same way as the training data

        X_train = X_train_scaled
        X_test = X_test_scaled

#Apply log transformation log(1+x) on target due to distribution skew
y_train_log = np.log1p(y_train) 

In [5]:
#Build model architecture
model = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(7,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(1)
])

#Define loss
model.compile(
    optimizer='adam',
    loss='mse',           # Use MSE
    metrics=['mae']
)

In [ ]:
#Train
model.fit(
    X_train, y_train_log,
    epochs=5,
    batch_size=512,
    validation_split=0.2,  # uses 20% of training data to monitor overfitting
    verbose=1,
    callbacks=[
        keras.callbacks.EarlyStopping(
            patience=5,              # stops if val_loss doesn't improve for 5 epochs
            restore_best_weights=True # rolls back to the best epoch
        )
    ],
)

Epoch 1/5
  439/48375 ━━━━━━━━━━━━━━━━━━━━ 1:12 2ms/step - loss: 6.9180 - mae: 1.8039

In [39]:
pred_log = model.predict(X_test)
pred = np.expm1(pred_log) #Undo log(x+1) transformation 
MAE = mean_absolute_error(y_test, pred)
print(f"MAE: {MAE:.4f}")

107500/107500 ━━━━━━━━━━━━━━━━━━━━ 56s 525us/step
MAE: 228.5450


In [23]:
#Evaluate
loss, MAE = model.evaluate(X_test, y_test)

# MAE = mean_absolute_error(y_test, y_pred)
# MSE = mean_squared_error(y_test, y_pred)

#print(f"MAE: {MAE:.4f}, MSE: {MSE:.4f}, RMSE: {pow(MSE,0.5):.4f}")
print(f"MAE: {MAE:.4f}")

107500/107500 ━━━━━━━━━━━━━━━━━━━━ 76s 703us/step - loss: 125737.8125 - mae: 232.9934
MAE: 232.9934


In [40]:
model.save("taxi_model_NN.keras")

In [42]:
import joblib
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']